In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')

print("spark session created")

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getO

spark session created


order_id,customer_name,product,amount,order_date
ORD-0001,sarah johnson,Webcam,132.63,2024-11-17

In [14]:
# READING  INTO DATAFRAME WITHOUT PROVIDING SCHEMA
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("s3a://glue-spark-etl-example-source/raw/orders/orders61.csv")

df.printSchema()

df.show()


root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)

+--------+--------------+----------+-------+----------+
|order_id| customer_name|   product| amount|order_date|
+--------+--------------+----------+-------+----------+
|ORD-0001| sarah johnson|    Webcam| 132.63|2024-11-17|
|ORD-0002|   emily davis|     Mouse|1227.21|2024-05-13|
|ORD-0003|    john smith|Headphones| 476.62|2024-03-22|
|ORD-0004|michael wilson|  Keyboard|    abc|2024-10-11|
|ORD-0005|      jane doe|    Tablet|   null|2024-09-22|
|ORD-0006|     david lee|     Phone|    N/A|2024-07-15|
+--------+--------------+----------+-------+----------+



In [ ]:
# READING  INTO DATAFRAME BY INERRING SCHEMA
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3a://glue-spark-etl-example-source/raw/orders/orders61.csv")

df.printSchema()

df.show()


root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: timestamp (nullable = true)

+--------+--------------+----------+-------+-------------------+
|order_id| customer_name|   product| amount|         order_date|
+--------+--------------+----------+-------+-------------------+
|ORD-0001| sarah johnson|    Webcam| 132.63|2024-11-17 00:00:00|
|ORD-0002|   emily davis|     Mouse|1227.21|2024-05-13 00:00:00|
|ORD-0003|    john smith|Headphones| 476.62|2024-03-22 00:00:00|
|ORD-0004|michael wilson|  Keyboard|    abc|2024-10-11 00:00:00|
|ORD-0005|      jane doe|    Tablet|   null|2024-09-22 00:00:00|
|ORD-0006|     david lee|     Phone|    N/A|2024-07-15 00:00:00|
+--------+--------------+----------+-------+-------------------+



In [17]:
# CREATING DATAFRAME BY PROVIDING SCHEMA EXPLICITLY
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

schema = StructType([
    StructField("order_id",      StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("product",       StringType(), True),
    StructField("amount",        DoubleType(), True),
    StructField("order_date",    DateType(),   True)
])

df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("s3a://glue-spark-etl-example-source/raw/orders/orders61.csv")

df.printSchema()
df.show()


root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: date (nullable = true)

+--------+--------------+----------+-------+----------+
|order_id| customer_name|   product| amount|order_date|
+--------+--------------+----------+-------+----------+
|ORD-0001| sarah johnson|    Webcam| 132.63|2024-11-17|
|ORD-0002|   emily davis|     Mouse|1227.21|2024-05-13|
|ORD-0003|    john smith|Headphones| 476.62|2024-03-22|
|ORD-0004|michael wilson|  Keyboard|   null|2024-10-11|
|ORD-0005|      jane doe|    Tablet|   null|2024-09-22|
|ORD-0006|     david lee|     Phone|   null|2024-07-15|
+--------+--------------+----------+-------+----------+



In [18]:
# Read source CSV
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type='s3',
    connection_options={'paths': ['s3a://glue-spark-etl-example-source/raw/orders/orders61.csv']},
    format='csv',
    format_options={'withHeader': True, 'separator': ','}
)
dyf.printSchema()
dyf.show()

root
|-- order_id: string
|-- customer_name: string
|-- product: string
|-- amount: string
|-- order_date: string

{"order_id": "ORD-0001", "customer_name": "sarah johnson", "product": "Webcam", "amount": "132.63", "order_date": "2024-11-17"}
{"order_id": "ORD-0002", "customer_name": "emily davis", "product": "Mouse", "amount": "1227.21", "order_date": "2024-05-13"}
{"order_id": "ORD-0003", "customer_name": "john smith", "product": "Headphones", "amount": "476.62", "order_date": "2024-03-22"}
{"order_id": "ORD-0004", "customer_name": "michael wilson", "product": "Keyboard", "amount": "abc", "order_date": "2024-10-11"}
{"order_id": "ORD-0005", "customer_name": "jane doe", "product": "Tablet", "amount": "", "order_date": "2024-09-22"}
{"order_id": "ORD-0006", "customer_name": "david lee", "product": "Phone", "amount": "N/A", "order_date": "2024-07-15"}


In [25]:
dyf_methods = [m for m in dir(dyf) if not m.startswith('_')]
dyf_methods

['applyMapping',
 'apply_mapping',
 'assertErrorThreshold',
 'coalesce',
 'count',
 'drop_fields',
 'errorsAsDynamicFrame',
 'errorsCount',
 'filter',
 'fromDF',
 'getNumPartitions',
 'glue_ctx',
 'join',
 'map',
 'mapPartitions',
 'mapPartitionsWithIndex',
 'mergeDynamicFrame',
 'name',
 'printSchema',
 'relationalize',
 'rename_field',
 'repartition',
 'resolveChoice',
 'schema',
 'select_fields',
 'show',
 'spigot',
 'split_fields',
 'split_rows',
 'stageErrorsCount',
 'toDF',
 'unbox',
 'union',
 'unnest',
 'unnest_ddb_json',
 'with_frame_schema',
 'write']

## 1. applyMapping — rename, reorder and cast columns

In [30]:
# (source_name, source_type, target_name, target_type)
mapped = dyf.apply_mapping([
    ('order_id',       'string', 'order_id',       'string'),
    ('customer_name',  'string', 'customer',       'string'),  # renamed
    ('product',        'string', 'product',        'string'),
    ('amount',         'string', 'amount',         'double'),  # cast to double
    ('order_date',     'string', 'order_date',     'date')     # cast to date
])
mapped.printSchema()
mapped.show()

root
|-- order_id: string
|-- customer: string
|-- product: string
|-- amount: double
|-- order_date: date

{"order_id": "ORD-0001", "customer": "sarah johnson", "product": "Webcam", "amount": 132.63, "order_date": 2024-11-17}
{"order_id": "ORD-0002", "customer": "emily davis", "product": "Mouse", "amount": 1227.21, "order_date": 2024-05-13}
{"order_id": "ORD-0003", "customer": "john smith", "product": "Headphones", "amount": 476.62, "order_date": 2024-03-22}
{"order_id": "ORD-0004", "customer": "michael wilson", "product": "Keyboard", "amount": null, "order_date": 2024-10-11}
{"order_id": "ORD-0005", "customer": "jane doe", "product": "Tablet", "amount": null, "order_date": 2024-09-22}
{"order_id": "ORD-0006", "customer": "david lee", "product": "Phone", "amount": null, "order_date": 2024-07-15}


## 2. resolveChoice — handle ambiguous or mixed-type columns

In [40]:
# cast  — force a specific type
resolved_cast = dyf.resolveChoice(
    specs=[('amount', 'cast:double')]
)
resolved_cast.printSchema()
resolved_cast.show()
resolved_cast.toDF().show()

root
|-- order_id: string
|-- customer_name: string
|-- product: string
|-- amount: double
|-- order_date: string

{"order_id": "ORD-0001", "customer_name": "sarah johnson", "product": "Webcam", "order_date": "2024-11-17", "amount": 132.63}
{"order_id": "ORD-0002", "customer_name": "emily davis", "product": "Mouse", "order_date": "2024-05-13", "amount": 1227.21}
{"order_id": "ORD-0003", "customer_name": "john smith", "product": "Headphones", "order_date": "2024-03-22", "amount": 476.62}
{"order_id": "ORD-0004", "customer_name": "michael wilson", "product": "Keyboard", "order_date": "2024-10-11"}
{"order_id": "ORD-0005", "customer_name": "jane doe", "product": "Tablet", "order_date": "2024-09-22"}
{"order_id": "ORD-0006", "customer_name": "david lee", "product": "Phone", "order_date": "2024-07-15"}
+--------+--------------+----------+-------+----------+
|order_id| customer_name|   product| amount|order_date|
+--------+--------------+----------+-------+----------+
|ORD-0001| sarah johnso

In [39]:
# make_struct — keep all types as a struct (useful when column has mixed types)
resolved_struct = dyf.resolveChoice(
    specs=[('amount', 'make_struct')]
)
resolved_struct.printSchema()
resolved_struct.show()
resolved_struct.toDF().show()

root
|-- order_id: string
|-- customer_name: string
|-- product: string
|-- amount: struct
|    |-- string: string
|-- order_date: string

{"order_id": "ORD-0001", "customer_name": "sarah johnson", "product": "Webcam", "order_date": "2024-11-17", "amount": {"string": "132.63"}}
{"order_id": "ORD-0002", "customer_name": "emily davis", "product": "Mouse", "order_date": "2024-05-13", "amount": {"string": "1227.21"}}
{"order_id": "ORD-0003", "customer_name": "john smith", "product": "Headphones", "order_date": "2024-03-22", "amount": {"string": "476.62"}}
{"order_id": "ORD-0004", "customer_name": "michael wilson", "product": "Keyboard", "order_date": "2024-10-11", "amount": {"string": "abc"}}
{"order_id": "ORD-0005", "customer_name": "jane doe", "product": "Tablet", "order_date": "2024-09-22", "amount": {"string": ""}}
{"order_id": "ORD-0006", "customer_name": "david lee", "product": "Phone", "order_date": "2024-07-15", "amount": {"string": "N/A"}}
+--------+--------------+----------+-----

In [ ]:

# project — pick one type and drop others
resolved_project = dyf.resolveChoice(
    specs=[('amount', 'project:double')]
)
resolved_project.printSchema()
resolved_project.show()
resolved_project.toDF().show()

root
|-- order_id: string
|-- customer_name: string
|-- product: string
|-- amount: double
|-- order_date: string

{"order_id": "ORD-0001", "customer_name": "sarah johnson", "product": "Webcam", "order_date": "2024-11-17", "amount": 132.63}
{"order_id": "ORD-0002", "customer_name": "emily davis", "product": "Mouse", "order_date": "2024-05-13", "amount": 1227.21}
{"order_id": "ORD-0003", "customer_name": "john smith", "product": "Headphones", "order_date": "2024-03-22", "amount": 476.62}
{"order_id": "ORD-0004", "customer_name": "michael wilson", "product": "Keyboard", "order_date": "2024-10-11"}
{"order_id": "ORD-0005", "customer_name": "jane doe", "product": "Tablet", "order_date": "2024-09-22"}
{"order_id": "ORD-0006", "customer_name": "david lee", "product": "Phone", "order_date": "2024-07-15"}
+--------+--------------+----------+-------+----------+
|order_id| customer_name|   product| amount|order_date|
+--------+--------------+----------+-------+----------+
|ORD-0001| sarah johnso

## 3. drop_fields — drop unwanted columns

In [44]:
dropped = mapped.drop_fields(['order_date'])
dropped.printSchema()
dropped.show()

root
|-- order_id: string
|-- customer: string
|-- product: string
|-- amount: double

{"order_id": "ORD-0001", "customer": "sarah johnson", "product": "Webcam", "amount": 132.63}
{"order_id": "ORD-0002", "customer": "emily davis", "product": "Mouse", "amount": 1227.21}
{"order_id": "ORD-0003", "customer": "john smith", "product": "Headphones", "amount": 476.62}
{"order_id": "ORD-0004", "customer": "michael wilson", "product": "Keyboard", "amount": null}
{"order_id": "ORD-0005", "customer": "jane doe", "product": "Tablet", "amount": null}
{"order_id": "ORD-0006", "customer": "david lee", "product": "Phone", "amount": null}


## 4. select_fields — keep only specific columns

In [45]:
selected = mapped.select_fields(['order_id', 'customer', 'amount'])
selected.printSchema()
selected.show(6)

root
|-- order_id: string
|-- customer: string
|-- amount: double

{"order_id": "ORD-0001", "customer": "sarah johnson", "amount": 132.63}
{"order_id": "ORD-0002", "customer": "emily davis", "amount": 1227.21}
{"order_id": "ORD-0003", "customer": "john smith", "amount": 476.62}
{"order_id": "ORD-0004", "customer": "michael wilson", "amount": null}
{"order_id": "ORD-0005", "customer": "jane doe", "amount": null}
{"order_id": "ORD-0006", "customer": "david lee", "amount": null}


## 5. filter — filter rows using a lambda

In [47]:
high_value = mapped.filter(f=lambda row: row['amount'] > 400)
print(f'High value orders: {high_value.count()}')
high_value.show(3)

High value orders: 2
{"product": "Mouse", "amount": 1227.21, "order_id": "ORD-0002", "customer": "emily davis", "order_date": 2024-05-13}
{"product": "Headphones", "amount": 476.62, "order_id": "ORD-0003", "customer": "john smith", "order_date": 2024-03-22}


## 6. map — apply a transformation to each row

In [48]:
def uppercase_customer(rec):
    rec['customer'] = rec['customer'].upper()
    return rec

uppercased = mapped.map(f=uppercase_customer)
uppercased.show(3)

{"product": "Webcam", "amount": 132.63, "order_id": "ORD-0001", "customer": "SARAH JOHNSON", "order_date": 2024-11-17}
{"product": "Mouse", "amount": 1227.21, "order_id": "ORD-0002", "customer": "EMILY DAVIS", "order_date": 2024-05-13}
{"product": "Headphones", "amount": 476.62, "order_id": "ORD-0003", "customer": "JOHN SMITH", "order_date": 2024-03-22}


## 7. rename_field — rename a single column

In [49]:
renamed = mapped.rename_field('customer', 'customer_name')
renamed.printSchema()

root
|-- order_id: string
|-- product: string
|-- amount: double
|-- order_date: date
|-- customer_name: string



## 8. Write transformed DynamicFrame back to MinIO

In [ ]:
glueContext.write_dynamic_frame.from_options(
    frame=mapped,
    connection_type='s3',
    connection_options={'path': 's3a://glue-spark-etl-example-target/silver/orders/'},
    format='parquet'
)
print('Write completed successfully')

In [ ]:
# spark.stop()